# 00.1 Python for ML 快速回顾

这份 notebook 不从头重学 Python，而是只复习后续 `PyTorch` 高频会用到的部分。  

本节重点

- 函数参数（Function arguments）
- 类、继承与 `super()`
- 特殊方法 `__len__` 与 `__getitem__`
- 上下文管理、断言、异常

学习方式

1. 先读题并预测结果
2. 再运行示例代码
3. 自己完成 `TODO` 练习
4. 最后再看参考答案

## 学习目标

学完后你应该能

1. 看懂训练脚本里的函数参数设计
2. 理解 `nn.Module` 和 `Dataset` 背后的 Python 基础
3. 独立写出带 `__len__` 和 `__getitem__` 的简单类
4. 用 `assert` 和异常做基础输入检查
5. 把这些 Python 基础映射到后面的 `PyTorch` 用法

## 1. 函数参数

在机器学习代码里，函数参数设计几乎无处不在。  

常见场景

- 传入超参数（Passing hyperparameters）
- 提供默认值（Providing default values）
- 通过关键字参数提高可读性
- 接收灵活配置（Accepting flexible configuration values）

In [ ]:
def build_experiment_name(model_name, lr=1e-3, batch_size=32, *, seed=42, extra_tags=None):
    """构造实验名 / Build a readable experiment name."""
    if extra_tags is None:
        extra_tags = []

    tag_text = "-".join(extra_tags) if extra_tags else "base"
    return f"{model_name}_lr{lr}_bs{batch_size}_seed{seed}_{tag_text}"


name = build_experiment_name(
    "mlp",
    batch_size=64,
    seed=7,
    extra_tags=["aug", "dropout"],
)

print("实验名 / experiment name:", name)

# 思考
# 为什么 seed 前面写了 * 之后，调用时必须显式写 seed=7 ？
# Why must seed be passed as seed=7 after the * marker?

关键点

- 默认参数
- 关键字专用参数
- Mutable default pitfall: `extra_tags=None` 比 `extra_tags=[]` 更安全  
  `extra_tags=None` is safer than `extra_tags=[]`.

这是训练函数中非常常见的模式。  


In [ ]:
# 练习 1
# 实现 summarize_split，返回一个字典
#
# 需要包含
# - train_size
# - val_size
# - test_size
# - total_size
# - shuffle
# - stratify
#
# 要求
# 1. test_size 默认是 0
# 2. shuffle 和 stratify 必须是关键字参数
# 3. 返回 dict

def summarize_split(train_size, val_size, test_size=0, *, shuffle=True, stratify=False):
    # TODO: 写你的实现
    pass


# print(summarize_split(800, 100, test_size=100, shuffle=True, stratify=True))

In [ ]:
# 练习 1 参考答案

def summarize_split_solution(train_size, val_size, test_size=0, *, shuffle=True, stratify=False):
    total_size = train_size + val_size + test_size
    return {
        "train_size": train_size,
        "val_size": val_size,
        "test_size": test_size,
        "total_size": total_size,
        "shuffle": shuffle,
        "stratify": stratify,
    }


print(summarize_split_solution(800, 100, test_size=100, shuffle=True, stratify=True))

### `*args` 与 `**kwargs`

你不必滥用它们，但必须能读懂。  

- `*args`：额外的位置参数（extra positional arguments）
- `**kwargs`：额外的关键字参数（extra keyword arguments）

In [ ]:
def log_metrics(epoch, *metric_values, **named_metrics):
    print(f"epoch={epoch}")
    print("位置参数指标 / positional metrics:", metric_values)
    print("命名指标 / named metrics:", named_metrics)


log_metrics(3, 0.91, 0.27, loss=0.27, accuracy=0.91)

## 2. 类、继承与 `super()`

`PyTorch` 的模型和数据集写法，本质上高度依赖面向对象。  

你至少要熟悉

- `__init__` 初始化对象
- 实例属性（instance attributes）
- 子类继承父类
- `super()` 调用父类逻辑（using `super()` to call parent logic）

In [ ]:
class MetricTracker:
    def __init__(self, name):
        self.name = name
        self.values = []

    def update(self, value):
        self.values.append(float(value))

    def compute(self):
        if not self.values:
            return 0.0
        return sum(self.values) / len(self.values)


class RunningAverage(MetricTracker):
    def __init__(self, name):
        super().__init__(name)
        self.total = 0.0
        self.count = 0

    def update(self, value):
        value = float(value)
        self.total += value
        self.count += 1
        self.values.append(value)

    def compute(self):
        if self.count == 0:
            return 0.0
        return self.total / self.count


loss_tracker = RunningAverage("train_loss")
for loss in [0.95, 0.72, 0.51]:
    loss_tracker.update(loss)

print("名称 / name:", loss_tracker.name)
print("平均值 / average:", loss_tracker.compute())

你需要读懂下面的映射

- Parent class: 放公共逻辑（stores shared logic）
- Child class: 扩展或重写行为（extends or overrides behavior）
- `super().__init__(...)`: 先把父类该做的初始化做完

后面写 `nn.Module` 时几乎也是同样的结构。  


In [ ]:
# 练习 2
# 实现一个 BatchCounter
#
# 目标
# - 记录 batch 数
# - 记录样本总数
# - 计算平均 batch 大小

class BatchCounter:
    def __init__(self):
        # TODO
        pass

    def update(self, batch):
        # TODO
        pass

    def summary(self):
        # TODO
        pass


# counter = BatchCounter()
# counter.update([1, 2, 3, 4])
# counter.update([5, 6])
# print(counter.summary())

In [ ]:
# 练习 2 参考答案

class BatchCounterSolution:
    def __init__(self):
        self.num_batches = 0
        self.num_samples = 0

    def update(self, batch):
        self.num_batches += 1
        self.num_samples += len(batch)

    def summary(self):
        avg_batch_size = 0.0 if self.num_batches == 0 else self.num_samples / self.num_batches
        return {
            "num_batches": self.num_batches,
            "num_samples": self.num_samples,
            "avg_batch_size": avg_batch_size,
        }


counter = BatchCounterSolution()
counter.update([1, 2, 3, 4])
counter.update([5, 6])
print(counter.summary())

## 3. `__len__` 与 `__getitem__`

这是本节最关键的部分之一，因为它直接对应 `PyTorch Dataset` 的最小接口。  

只要一个类实现了

- `__len__`
- `__getitem__`

它就能表现得像一个可索引的数据容器。  


In [ ]:
class ToyDataset:
    def __init__(self, samples):
        self.samples = list(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        features, label = self.samples[index]
        return {"x": features, "y": label}


samples = [
    ([0.1, 0.2], 0),
    ([0.7, 0.9], 1),
    ([0.3, 0.4], 0),
]

dataset = ToyDataset(samples)
print("样本数 / dataset length:", len(dataset))
print("第 1 个样本 / item 1:", dataset[1])

In [ ]:
# 练习 3
# 实现一个滑动窗口数据集
#
# 例子
# values = [10, 11, 12, 13, 14], window_size = 2
# 样本
# - x=[10, 11], y=12
# - x=[11, 12], y=13
# - x=[12, 13], y=14

class WindowDataset:
    def __init__(self, values, window_size):
        self.values = list(values)
        self.window_size = window_size

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = WindowDataset([10, 11, 12, 13, 14], window_size=2)
# print(len(ds))
# print(ds[0])

In [ ]:
# 练习 3 参考答案

class WindowDatasetSolution:
    def __init__(self, values, window_size):
        self.values = list(values)
        self.window_size = window_size

    def __len__(self):
        return len(self.values) - self.window_size

    def __getitem__(self, index):
        window = self.values[index : index + self.window_size]
        target = self.values[index + self.window_size]
        return {"x": window, "y": target}


ds = WindowDatasetSolution([10, 11, 12, 13, 14], window_size=2)
print("长度 / length:", len(ds))
print(ds[0])
print(ds[1])
print(ds[2])

## 4. 上下文管理、断言与异常

这部分不是边角语法，而是写稳健训练代码的重要工具。  

- `with`：安全地管理资源（safely manage resources）
- `assert`：快速做前置检查（perform quick sanity checks）
- `raise`：主动抛出清晰错误（raise clear errors deliberately）

In [ ]:
from io import StringIO


with StringIO("epoch=1,loss=0.82\nepoch=2,loss=0.64\n") as f:
    lines = [line.strip() for line in f if line.strip()]

print("日志行 / log lines:", lines)


def validate_batch(features, labels):
    assert len(features) == len(labels), "features 和 labels 的长度必须一致 / features and labels must have the same length"
    if len(features) == 0:
        raise ValueError("batch 不能为空 / batch must not be empty")
    return True


print("检查结果 / validation result:", validate_batch([[1, 2], [3, 4]], [0, 1]))

In [ ]:
# 练习 4
# 改错题
# 下面这段代码至少有两个 bug。/ The class below has at least two bugs.

class BrokenDataset:
    def __init__(self, rows):
        self.rows = rows

    def len(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.row[index]


# TODO:
# 1. 修复这个类
# 2. 创建实例并验证 len(ds) 和 ds[0]

In [ ]:
# 练习 4 参考答案

class FixedDataset:
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


fixed_ds = FixedDataset([("a", 0), ("b", 1), ("c", 0)])
print("长度 / length:", len(fixed_ds))
print("第一个样本 / first sample:", fixed_ds[0])

## 5. 综合小练习

把前面的内容组合起来：  

- 类（classes）
- `__len__` 与 `__getitem__`
- 参数设计（argument design）
- 输入检查（input validation）

In [ ]:
# 练习 5
# 实现一个最小版表格数据集
#
# 要求
# 1. __len__ 返回样本数
# 2. __getitem__ 返回 (features, target)
# 3. features 按 feature_keys 顺序提取
# 4. 如果 target_key 缺失，抛出 KeyError

records = [
    {"hours": 1.5, "attendance": 0.70, "passed": 0},
    {"hours": 3.0, "attendance": 0.90, "passed": 1},
    {"hours": 2.2, "attendance": 0.80, "passed": 1},
]


class MiniTabularDataset:
    def __init__(self, records, feature_keys, target_key):
        self.records = list(records)
        self.feature_keys = list(feature_keys)
        self.target_key = target_key

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = MiniTabularDataset(records, feature_keys=["hours", "attendance"], target_key="passed")
# print(len(ds))
# print(ds[0])

In [ ]:
# 练习 5 参考答案

class MiniTabularDatasetSolution:
    def __init__(self, records, feature_keys, target_key):
        self.records = list(records)
        self.feature_keys = list(feature_keys)
        self.target_key = target_key

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        if self.target_key not in row:
            raise KeyError(f"missing target key: {self.target_key}")

        features = [row[key] for key in self.feature_keys]
        target = row[self.target_key]
        return features, target


ds = MiniTabularDatasetSolution(records, feature_keys=["hours", "attendance"], target_key="passed")
print("长度 / length:", len(ds))
print(ds[0])
print(ds[1])

## 6. 小结

这一节最重要的是建立这些映射：  

- argument design -> 训练函数（training utilities）
- 类与继承
- `__len__`, `__getitem__` -> 数据集接口（dataset interface）
- exceptions -> 更稳健的训练代码（more robust training code）

你现在应该能回答

1. 为什么 `Dataset` 常实现 `__len__` 和 `__getitem__`？
2. 为什么不建议把列表直接写成默认参数？
3. `super()` 在子类初始化里到底做了什么？
4. 为什么训练代码里常先做断言检查？

下一步建议

- 进入 `NumPy` 核心，重点建立 `shape` 直觉（Move to the NumPy core notebook and build strong `shape` intuition.）